In [19]:
# ============================================================
# 1. Imports
# ============================================================

import os
import sys
import json
import joblib
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

sys.path.append('/host/d/Github/')

import Osteosarcoma.Build_lists.Build_list as Build_list
import Osteosarcoma.functions_collection as ff
import Osteosarcoma.Image_2D.Generator as Generator
import Osteosarcoma.Image_2D.resnet50.model as model_module

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE:', DEVICE)


DEVICE: cuda


In [20]:
# ============================================================
# 2. Define dataset: use random0 split, folds 0-5
# ============================================================

label = 'Prognosis'
label_col = label + '_label'
random_state = 0

patient_list_file = (
    '/host/e/D/Data/Habitats/Jishuitan/Patient_lists/'
    f'image_label_info_set12_5fold_{label.lower()}_random{random_state}.xlsx'
)

data_root = '/host/e/D/Data/Habitats/Jishuitan/resampled_data_new'
out_root = '/host/d/projects/Habitats/radiomics/dl_2d_ml'
feature_np_out_dir = os.path.join(out_root, 'features_numpy')

ff.make_folder([out_root, feature_np_out_dir])

print('patient_list_file:', patient_list_file)
print('data_root:', data_root)
print('out_root:', out_root)

build = Build_list.Build(patient_list_file)


def build_case_df(batch_list):
    fold_list, patient_set_list, patient_index_list, label_list, _, _ = build.__build__(
        batch_list=batch_list,
        label_column_name=label_col,
    )

    rows = []
    for i in range(len(patient_index_list)):
        patient_set = str(patient_set_list[i])
        patient_index = str(patient_index_list[i])
        image_path = os.path.join(data_root, patient_set, patient_index, 'img_slices.nii.gz')
        mask_path = os.path.join(data_root, patient_set, patient_index, 'label_slices.nii.gz')
        bbox_path = os.path.join(data_root, patient_set, patient_index, 'bbox_mask_slices.nii.gz')
        rows.append({
            'Patient_set': patient_set,
            'Patient_index': patient_index,
            'fold': int(fold_list[i]),
            'Label': int(label_list[i]),
            'Image_filepath': image_path,
            'Mask_filepath': mask_path,
            'BBox_filepath': bbox_path,
        })
    return pd.DataFrame(rows)

fold_case_dfs = {fold: build_case_df([fold]) for fold in [0, 1, 2, 3, 4, 5]}
all_case_df = pd.concat([fold_case_dfs[fold] for fold in [0, 1, 2, 3, 4, 5]], ignore_index=True)

print('All cases:', all_case_df.shape)
print(all_case_df.groupby('fold')['Label'].agg(['count', 'mean']))
all_case_df.head()


patient_list_file: /host/e/D/Data/Habitats/Jishuitan/Patient_lists/image_label_info_set12_5fold_prognosis_random0.xlsx
data_root: /host/e/D/Data/Habitats/Jishuitan/resampled_data_new
out_root: /host/d/projects/Habitats/radiomics/dl_2d_ml
All cases: (330, 7)
      count      mean
fold                 
0        47  0.297872
1        47  0.297872
2        46  0.282609
3        46  0.304348
4        46  0.304348
5        98  0.295918


,Patient_set,Patient_index,fold,Label,Image_filepath,Mask_filepath,BBox_filepath
0,set_1,7,0,1,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...
1,set_1,22,0,0,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...
2,set_1,26,0,1,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...
3,set_1,33,0,0,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...
4,set_1,38,0,1,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...


In [21]:
# ============================================================
# 3. Manually define which model is used for each fold
# ============================================================
# Fill these paths before running feature extraction.
# Convention: fold k cases should normally use the model trained with val_fold=k,
# so features for folds 0-4 are out-of-fold DL features.

model_depth = 18          # 18, 34, or 50
input_mode = '2.5d'       # '2d' or '2.5d'
dropout_p = 0.3          # Must match the trained ResNet head: fc = Dropout + Linear
image_size = (144, 144)
batch_size = 16

# 现在版本的模型路径都是一样的， 0: '/host/d/projects/Habitats/models/Prognosis/resnet18_2.5D_FTall_AUGfull_sgd/random0_all_fold5/models/model-35.pt',
fold_model_paths = {
    0: '/host/d/projects/Habitats/models/Prognosis/resnet18_2.5D_FTall_AUGfull_sgd/random0_all_fold5/models/model-35.pt',
    1: '/host/d/projects/Habitats/models/Prognosis/resnet18_2.5D_FTall_AUGfull_sgd/random0_all_fold5/models/model-35.pt',
    2: '/host/d/projects/Habitats/models/Prognosis/resnet18_2.5D_FTall_AUGfull_sgd/random0_all_fold5/models/model-35.pt',
    3: '/host/d/projects/Habitats/models/Prognosis/resnet18_2.5D_FTall_AUGfull_sgd/random0_all_fold5/models/model-35.pt',
    4: '/host/d/projects/Habitats/models/Prognosis/resnet18_2.5D_FTall_AUGfull_sgd/random0_all_fold5/models/model-35.pt',
}

for fold, path in fold_model_paths.items():
    print(f'fold{fold}_model_path:', path)


fold0_model_path: /host/d/projects/Habitats/models/Prognosis/resnet18_2.5D_FTall_AUGfull_sgd/random0_all_fold5/models/model-35.pt
fold1_model_path: /host/d/projects/Habitats/models/Prognosis/resnet18_2.5D_FTall_AUGfull_sgd/random0_all_fold5/models/model-35.pt
fold2_model_path: /host/d/projects/Habitats/models/Prognosis/resnet18_2.5D_FTall_AUGfull_sgd/random0_all_fold5/models/model-35.pt
fold3_model_path: /host/d/projects/Habitats/models/Prognosis/resnet18_2.5D_FTall_AUGfull_sgd/random0_all_fold5/models/model-35.pt
fold4_model_path: /host/d/projects/Habitats/models/Prognosis/resnet18_2.5D_FTall_AUGfull_sgd/random0_all_fold5/models/model-35.pt


In [22]:
# ============================================================
# Helper functions: model loading and avgpool feature extraction
# ============================================================

def load_resnet_checkpoint(model, checkpoint_path):
    if checkpoint_path is None or str(checkpoint_path).strip() == '':
        raise ValueError('checkpoint_path is empty. Please fill fold_model_paths first.')
    if not os.path.isfile(checkpoint_path):
        raise FileNotFoundError(checkpoint_path)

    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    state_dict = checkpoint['model'] if isinstance(checkpoint, dict) and 'model' in checkpoint else checkpoint

    cleaned = {}
    for key, value in state_dict.items():
        if key.startswith('module.'):
            key = key[len('module.'):]
        cleaned[key] = value

    model.load_state_dict(cleaned, strict=True)
    return model


def build_loaded_model(checkpoint_path):
    # use_imagenet=False because the checkpoint contains the trained weights.
    model = model_module.build_resnet_model(
        model_depth=model_depth,
        num_classes=2,
        use_imagenet=False,
        dropout_p=dropout_p,
    )
    model = load_resnet_checkpoint(model, checkpoint_path)
    model.to(DEVICE)
    model.eval()
    return model


def resnet_avgpool_features(model, x):
    """Return final avgpool vector from a torchvision ResNet."""
    x = model.conv1(x)
    x = model.bn1(x)
    x = model.relu(x)
    x = model.maxpool(x)
    x = model.layer1(x)
    x = model.layer2(x)
    x = model.layer3(x)
    x = model.layer4(x)
    x = model.avgpool(x)
    x = torch.flatten(x, 1)
    return x


def make_dataset(case_df):
    return Generator.Dataset_2D(
        patient_set_list=case_df['Patient_set'].astype(str).tolist(),
        patient_index_list=case_df['Patient_index'].astype(str).tolist(),
        x_file_list=case_df['Image_filepath'].astype(str).tolist(),
        y_list=case_df['Label'].astype(int).tolist(),
        data_root=data_root,
        target_image_size=image_size,
        normalize_factor='equation',
        only_tumor_pixels='roi',
        augment_context='simple',
        shuffle=False,
        augment=False,
        augment_frequency=0,
    )


def extract_features_for_cases(model, case_df):
    dataset = make_dataset(case_df)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)

    features = []
    labels = []

    model.eval()
    with torch.no_grad():
        for batch in loader:
            slice_front, slice_middle, slice_rear, batch_y = batch

            if input_mode == '2d':
                x = slice_middle.to(DEVICE)
                feat = resnet_avgpool_features(model, x)
            elif input_mode == '2.5d':
                feat_front = resnet_avgpool_features(model, slice_front.to(DEVICE))
                feat_middle = resnet_avgpool_features(model, slice_middle.to(DEVICE))
                feat_rear = resnet_avgpool_features(model, slice_rear.to(DEVICE))
                feat = (feat_front + feat_middle + feat_rear) / 3.0
            else:
                raise ValueError(f'Unsupported input_mode: {input_mode}')

            features.append(feat.detach().cpu().numpy())
            labels.extend(batch_y.cpu().numpy().astype(int).tolist())

    features = np.concatenate(features, axis=0) if len(features) > 0 else np.zeros((0, 0), dtype=np.float32)
    labels = np.asarray(labels).astype(int)

    if len(case_df) != features.shape[0]:
        raise RuntimeError(f'Feature number mismatch: cases={len(case_df)}, features={features.shape[0]}')

    return features.astype(np.float32), labels


In [23]:
# ============================================================
# 4. Extract DL features for folds 0-4 using their corresponding models
# ============================================================

fold01234_feature_list = []
fold01234_meta_list = []

for fold in [0, 1, 2, 3, 4]:
    print('\n============================================================')
    print('Extracting fold:', fold)
    case_df = fold_case_dfs[fold].copy().reset_index(drop=True)
    model = build_loaded_model(fold_model_paths[fold])
    features, labels_check = extract_features_for_cases(model, case_df)

    if not np.array_equal(labels_check, case_df['Label'].astype(int).to_numpy()):
        raise RuntimeError(f'Label mismatch during feature extraction for fold {fold}.')

    fold01234_feature_list.append(features)
    fold01234_meta_list.append(case_df)

    np.save(os.path.join(feature_np_out_dir, f'fold{fold}_DLfeature.npy'), features)
    case_df.to_excel(os.path.join(feature_np_out_dir, f'fold{fold}_metadata.xlsx'), index=False)

fold01234_DLfeature = np.concatenate(fold01234_feature_list, axis=0)
fold01234_metadata = pd.concat(fold01234_meta_list, ignore_index=True)

np.save(os.path.join(feature_np_out_dir, 'fold01234_DLfeature.npy'), fold01234_DLfeature)
fold01234_metadata.to_excel(os.path.join(feature_np_out_dir, 'fold01234_metadata.xlsx'), index=False)

print('fold01234 feature shape:', fold01234_DLfeature.shape)
fold01234_metadata.head()



Extracting fold: 0

Extracting fold: 1

Extracting fold: 2

Extracting fold: 3

Extracting fold: 4
fold01234 feature shape: (232, 512)


,Patient_set,Patient_index,fold,Label,Image_filepath,Mask_filepath,BBox_filepath
0,set_1,7,0,1,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...
1,set_1,22,0,0,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...
2,set_1,26,0,1,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...
3,set_1,33,0,0,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...
4,set_1,38,0,1,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...


In [24]:
# ============================================================
# 5. Extract DL features for fold 5 using all five fold models
# ============================================================

fold5_df = fold_case_dfs[5].copy().reset_index(drop=True)
fold5_features_by_model = {}

for model_fold in [0, 1, 2, 3, 4]:
    print('\n============================================================')
    print('Extracting fold5 with model from fold:', model_fold)
    model = build_loaded_model(fold_model_paths[model_fold])
    features, labels_check = extract_features_for_cases(model, fold5_df)

    if not np.array_equal(labels_check, fold5_df['Label'].astype(int).to_numpy()):
        raise RuntimeError(f'Label mismatch during fold5 extraction with model {model_fold}.')

    fold5_features_by_model[model_fold] = features
    np.save(os.path.join(feature_np_out_dir, f'fold5_DLfeature_model{model_fold}.npy'), features)

fold5_stack = np.stack([fold5_features_by_model[k] for k in [0, 1, 2, 3, 4]], axis=0)
fold5_DLfeature_avg = np.mean(fold5_stack, axis=0).astype(np.float32)

np.save(os.path.join(feature_np_out_dir, 'fold5_DLfeature_avg.npy'), fold5_DLfeature_avg)
fold5_df.to_excel(os.path.join(feature_np_out_dir, 'fold5_metadata.xlsx'), index=False)

print('fold5 one-model feature shape:', fold5_features_by_model[0].shape)
print('fold5 averaged feature shape:', fold5_DLfeature_avg.shape)



Extracting fold5 with model from fold: 0

Extracting fold5 with model from fold: 1

Extracting fold5 with model from fold: 2

Extracting fold5 with model from fold: 3

Extracting fold5 with model from fold: 4
fold5 one-model feature shape: (98, 512)
fold5 averaged feature shape: (98, 512)


In [25]:
# ============================================================
# 6. Manually choose which fold5 feature set to use
# ============================================================
# Options:
#   'avg' -> average of five model features
#   0,1,2,3,4 -> features extracted by one specific fold model

fold5_feature_choice = 0

if fold5_feature_choice == 'avg':
    fold5_DLfeature_selected = fold5_DLfeature_avg
elif fold5_feature_choice in [0, 1, 2, 3, 4]:
    fold5_DLfeature_selected = fold5_features_by_model[int(fold5_feature_choice)]
else:
    raise ValueError("fold5_feature_choice must be 'avg' or one of 0,1,2,3,4")

print('Selected fold5 feature:', fold5_feature_choice)
print('Selected fold5 feature shape:', fold5_DLfeature_selected.shape)


Selected fold5 feature: 0
Selected fold5 feature shape: (98, 512)


In [26]:
# ============================================================
# 7. PCA on raw DL features, then MinMax normalize PCA features
# ============================================================

from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
import joblib
import os
import json
import numpy as np
import pandas as pd


# ============================================================
# 1. Combine fold0-4 features and selected fold5 features
# ============================================================

all_DLfeature_raw = np.concatenate(
    [
        fold01234_DLfeature,
        fold5_DLfeature_selected,
    ],
    axis=0,
)

all_metadata = pd.concat(
    [
        fold01234_metadata,
        fold5_df,
    ],
    ignore_index=True,
)

print("All raw DL feature shape:", all_DLfeature_raw.shape)
print("All metadata shape:", all_metadata.shape)


# ============================================================
# 2. PCA directly on raw DL features
# ============================================================

pca_n_components = 0.95

pca = PCA(
    n_components=pca_n_components,
    random_state=random_state,
)

all_DLfeature_pca_raw = pca.fit_transform(all_DLfeature_raw)

print("Raw PCA feature shape:", all_DLfeature_pca_raw.shape)
print("Explained variance ratio sum:", float(np.sum(pca.explained_variance_ratio_)))
print("Raw PCA min:", float(np.min(all_DLfeature_pca_raw)))
print("Raw PCA max:", float(np.max(all_DLfeature_pca_raw)))


# ============================================================
# 3. MinMax normalize PCA features for final ML input
# ============================================================

pca_minmax_scaler = MinMaxScaler(feature_range=(0, 1))

all_DLfeature_pca = pca_minmax_scaler.fit_transform(all_DLfeature_pca_raw)

print("Final normalized PCA feature shape:", all_DLfeature_pca.shape)
print("Final PCA feature min:", float(np.min(all_DLfeature_pca)))
print("Final PCA feature max:", float(np.max(all_DLfeature_pca)))


# ============================================================
# 4. Save raw feature, raw PCA, normalized PCA, PCA model, and scaler
# ============================================================

os.makedirs(feature_np_out_dir, exist_ok=True)

np.save(
    os.path.join(feature_np_out_dir, "all_DLfeature_raw_selected.npy"),
    all_DLfeature_raw,
)

np.save(
    os.path.join(feature_np_out_dir, "all_DLfeature_PCA_raw.npy"),
    all_DLfeature_pca_raw,
)

np.save(
    os.path.join(feature_np_out_dir, "all_DLfeature_PCA_minmax.npy"),
    all_DLfeature_pca,
)

joblib.dump(
    pca,
    os.path.join(feature_np_out_dir, "DLfeature_pca.joblib"),
)

joblib.dump(
    pca_minmax_scaler,
    os.path.join(feature_np_out_dir, "DLfeature_pca_minmax_scaler.joblib"),
)


# ============================================================
# 5. Save settings
# ============================================================

settings = {
    "label": label,
    "random_state": random_state,
    "model_depth": model_depth,
    "input_mode": input_mode,
    "image_size": image_size,
    "fold_model_paths": fold_model_paths,
    "fold5_feature_choice": fold5_feature_choice,
    "raw_feature_normalization": "None",
    "pca_input": "raw DL features",
    "pca_n_components": pca_n_components,
    "pca_output_components": int(all_DLfeature_pca.shape[1]),
    "pca_explained_variance_ratio_sum": float(np.sum(pca.explained_variance_ratio_)),
    "final_feature_normalization": "MinMaxScaler fitted on PCA features; final ML input is [0,1]",
}

with open(
    os.path.join(feature_np_out_dir, "feature_extraction_settings.json"),
    "w",
    encoding="utf-8",
) as f:
    json.dump(settings, f, indent=2)

All raw DL feature shape: (330, 512)
All metadata shape: (330, 7)
Raw PCA feature shape: (330, 145)
Explained variance ratio sum: 0.9503704309463501
Raw PCA min: -10.676586151123047
Raw PCA max: 23.78902244567871
Final normalized PCA feature shape: (330, 145)
Final PCA feature min: 0.0
Final PCA feature max: 1.0000001192092896


In [27]:
# ============================================================
# 8. Save final MinMax-normalized PCA DL features as Excel table
# ============================================================

feature_columns = [
    f"DL_feature_{i+1:03d}"
    for i in range(all_DLfeature_pca.shape[1])
]

feature_df = pd.DataFrame(
    all_DLfeature_pca,
    columns=feature_columns,
)

metadata_columns = [
    "Patient_set",
    "Patient_index",
    "Image_filepath",
    "Mask_filepath",
    "fold",
    "Label",
]

ml_table_df = pd.concat(
    [
        all_metadata[metadata_columns].reset_index(drop=True),
        feature_df.reset_index(drop=True),
    ],
    axis=1,
)

save_path = os.path.join(
    out_root,
    "dl_2d_features_PCA.xlsx",
)

ml_table_df.to_excel(save_path, index=False)

print("Saved final MinMax-normalized PCA DL feature table:")
print(save_path)
print("Shape:", ml_table_df.shape)

feature_cols_check = [
    col for col in ml_table_df.columns
    if col.startswith("DL_feature_")
]

print("Final feature min:", float(ml_table_df[feature_cols_check].min().min()))
print("Final feature max:", float(ml_table_df[feature_cols_check].max().max()))

ml_table_df.head()

Saved final MinMax-normalized PCA DL feature table:
/host/d/projects/Habitats/radiomics/dl_2d_ml/dl_2d_features_PCA.xlsx
Shape: (330, 151)
Final feature min: 0.0
Final feature max: 1.0000001192092896


,Patient_set,Patient_index,Image_filepath,Mask_filepath,fold,Label,DL_feature_001,DL_feature_002,DL_feature_003,DL_feature_004,...,DL_feature_136,DL_feature_137,DL_feature_138,DL_feature_139,DL_feature_140,DL_feature_141,DL_feature_142,DL_feature_143,DL_feature_144,DL_feature_145
0,set_1,7,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,0,1,0.278502,0.131128,0.494064,0.496386,...,0.496333,0.462783,0.608152,0.639001,0.685048,0.574887,0.664835,0.335213,0.721534,0.332778
1,set_1,22,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,0,0,0.320549,0.498405,0.770780,0.152888,...,0.796232,0.535429,0.190131,0.411681,0.234220,0.781223,0.930817,0.733295,0.479119,0.390411
2,set_1,26,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,0,1,0.245951,0.364749,0.847790,0.169646,...,0.575697,0.445143,0.408504,0.544390,0.514046,0.724303,0.445119,0.460977,0.419341,0.350574
3,set_1,33,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,0,0,0.446520,0.615137,0.353041,0.182369,...,0.879767,0.273434,0.394684,0.554144,0.354751,0.495677,0.589365,0.473462,0.416827,0.633179
4,set_1,38,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,0,1,0.334840,0.315951,0.493902,0.052394,...,0.458990,0.294604,0.631645,0.379463,0.539095,0.709724,0.309916,0.597084,0.451388,0.675122
